# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR² Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Get dataset-level metadata (returns a Dict)
meta = dataset.metadata.to_json()
print(f"{meta['name']}: {meta['description']}")

## 2. Data Overview
Review all available record sets and their field ids using their `@id`s.

In [ ]:
# List all available record sets with their @id
record_sets = dataset.record_sets
print(f"Record sets found ({len(record_sets)}):\n")
rs_ids = []
for rs in record_sets:
    print(f"- RecordSet @id: {rs.metadata['@id']}")
    rs_ids.append(rs.metadata['@id'])
    print(f"  Name: {rs.metadata.get('name', '')}")
    print(f"  Description: {rs.metadata.get('description', '')}\n")

# For each record set, print its fields
for rs in record_sets:
    print(f"Fields for RecordSet (@id={rs.metadata['@id']}):")
    for field in rs.fields:
        print(f"  - @id: {field['@id']}  name: {field.get('name', '')}  dataType: {field.get('dataType','')}")
    print()

## 3. Data Extraction
Load data from available record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In this notebook, we will work with the **primary protein record set** (usually the main table containing protein annotations and quantifications; adjust @id to the record set of interest based on the cell above).

In [ ]:
# Assign the main record set @id.
# Update this to your specific record set @id as discovered in the previous step. Here we pick the first one by default for demonstration.
MAIN_RECORD_SET_ID = rs_ids[0]

all_dataframes = {}

for rs_id in rs_ids:
    # Collect records for this record set
    print(f"Loading data for RecordSet @id={rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    all_dataframes[rs_id] = df
    print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")

# Preview the main DataFrame
main_df = all_dataframes[MAIN_RECORD_SET_ID]
print(f"\nMain dataset columns:")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, and grouping. **All columns and fields must be referenced by their `@id`.**

We'll select a numeric field for demonstration. Based on the previous DataFrame columns, pick a numeric field such as molecular weight, peptide count, or abundance.

In [ ]:
# Example: Use numeric field by @id (adjust field name to match your main DataFrame's columns)
# You can reference any field seen in main_df.columns, e.g., 'cr:peptide_count' or 'cr:molecular_weight'
NUMERIC_FIELD_ID = None
for possible_id in main_df.columns:
    if 'weight' in possible_id.lower() or 'mw' in possible_id.lower():
        NUMERIC_FIELD_ID = possible_id
        break
if NUMERIC_FIELD_ID is None:
    for possible_id in main_df.columns:
        if 'count' in possible_id.lower():
            NUMERIC_FIELD_ID = possible_id
            break
if NUMERIC_FIELD_ID is None:
    NUMERIC_FIELD_ID = main_df.select_dtypes(include='number').columns[0] if len(main_df.select_dtypes(include='number').columns) > 0 else main_df.columns[0]

print(f"Using numeric field '@id': {NUMERIC_FIELD_ID}")

# Drop NA and filter
numeric_series = pd.to_numeric(main_df[NUMERIC_FIELD_ID], errors='coerce')
main_df = main_df.assign(**{NUMERIC_FIELD_ID: numeric_series})

threshold = main_df[NUMERIC_FIELD_ID].mean()

filtered_df = main_df[main_df[NUMERIC_FIELD_ID] > threshold]
print(f"Filtered records with {NUMERIC_FIELD_ID} > mean value ({threshold:.2f}): Found {len(filtered_df)} records")
print(filtered_df[[NUMERIC_FIELD_ID]].head())

# Normalize
filtered_df[f"{NUMERIC_FIELD_ID}_normalized"] = (filtered_df[NUMERIC_FIELD_ID] - filtered_df[NUMERIC_FIELD_ID].mean()) / filtered_df[NUMERIC_FIELD_ID].std()
print(f"\nNormalized {NUMERIC_FIELD_ID} for filtered records:")
print(filtered_df[[NUMERIC_FIELD_ID, f"{NUMERIC_FIELD_ID}_normalized"]].head())

# Try grouping by a field -- e.g., by protein accession or any categorical field containing 'accession' or 'id'
GROUP_FIELD_ID = None
for possible_id in main_df.columns:
    if 'accession' in possible_id.lower() or ('id' in possible_id.lower() and possible_id != NUMERIC_FIELD_ID):
        GROUP_FIELD_ID = possible_id
        break

if GROUP_FIELD_ID:
    grouped_df = filtered_df.groupby(GROUP_FIELD_ID)[NUMERIC_FIELD_ID].mean()
    print(f"\nGrouped data by {GROUP_FIELD_ID} (mean {NUMERIC_FIELD_ID}):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized version using a histogram and boxplot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
sns.histplot(filtered_df[NUMERIC_FIELD_ID].dropna(), kde=True)
plt.title(f"Distribution of {NUMERIC_FIELD_ID}")

plt.subplot(1,2,2)
sns.boxplot(y=filtered_df[NUMERIC_FIELD_ID])
plt.title(f"Boxplot of {NUMERIC_FIELD_ID}")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
sns.histplot(filtered_df[f"{NUMERIC_FIELD_ID}_normalized"].dropna(), kde=True)
plt.title(f"Normalized {NUMERIC_FIELD_ID}")
plt.xlabel(f"{NUMERIC_FIELD_ID}_normalized")
plt.show()

# Optionally, if you have a categorical group field, plot the means
if GROUP_FIELD_ID:
    grouped_plot_data = filtered_df.groupby(GROUP_FIELD_ID)[NUMERIC_FIELD_ID].mean().sort_values(ascending=False).head(15)
    plt.figure(figsize=(10,4))
    sns.barplot(x=grouped_plot_data.index, y=grouped_plot_data.values)
    plt.title(f"Top 15 {GROUP_FIELD_ID} by mean {NUMERIC_FIELD_ID}")
    plt.xlabel(GROUP_FIELD_ID)
    plt.ylabel(f"Mean {NUMERIC_FIELD_ID}")
    plt.xticks(rotation=75)
    plt.show()

## 6. Conclusion

- Loaded the FAIR² "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using `mlcroissant`.
- Explored the available record sets, fields, and columns by their `@id`.
- Demonstrated how to extract, filter, normalize, group, and visualize protein-centric data, referencing all entities by their `@id`.
- For more advanced analysis (clustering, machine learning, advanced statistical tests), see the rest of the [mlcroissant documentation](https://mlcommons.github.io/croissant/).
